## 第10章 生成器和推导式

### 1.生成器

- **生成器函数**：使用`yield`关键字定义的函数，返回一个生成器迭代器(又称生成器对象)。
    - 首先调用生成器函数，获取一个生成器迭代器。
    - 每次迭代时，生成器运行到`yield`语句，返回一个值，暂停执行，然后等待下次`__next__()`调用。
    - 再次调用`__next__()`时，生成器从暂停处精确恢复执行。

In [ ]:
from itertools import product

alphabet = 'ABCDEF'
registrations = {}

# 生成器函数
def gen_license_plates():
    for letters in product(alphabet, repeat=3): # 生成3个字母的所有组合
        letters = ''.join(letters)
        for numbers in range(1000):
            yield f'{letters}{numbers:03}'      # yield，返回一个值，然后暂停执行，等待下次调用

# 注册车牌号码
def new_registration(gen, owner):
    if owner not in registrations:
        plate = next(gen)                       # 从生成器中获取一个车牌号码
        registrations[owner] = plate
        return plate
    return None

license_plates = gen_license_plates()           # 创建生成器对象
for _ in range(888):                            # 先消耗部分数据
    next(license_plates)

name = "Jason C. McDonald"
my_plate = new_registration(license_plates, name)
print(my_plate)
print(registrations[name])
license_plates.close()

- **生成器VS迭代器：**
    - 迭代器的`__next__()`方法需显式抛出`StopIteration`来表示耗尽。
    - 生成器不需要显式抛出`StopIteration`，生成器终止时(到达末尾或`return`语句)，`StopIteration`在**幕后自动抛出**。

In [ ]:
# 迭代器类版本
from random import choice
colors = ['red', 'green', 'blue', 'silver', 'white', 'black']
vehicles = ['car', 'truck', 'semi', 'motorcycle', None]

class Traffic:
    def __iter__(self):                 # 可迭代
        return self

    def __next__(self):                 # 迭代器
        vehicle = choice(vehicles)      # 从车辆列表中随机选择一个
        if vehicle is None:
            raise StopIteration         # 如果选择到的是None，抛出StopIteration异常
        color = choice(colors)
        return f'{color} {vehicle}'

count = 0
for count, vehicle in enumerate(Traffic(), start=1):
    print(f"Wait for {vehicle}...")
print(f"Merged after {count} vehicles!")

In [ ]:
# 生成器版本
from random import choice
colors = ['red', 'green', 'blue', 'silver', 'white', 'black']
vehicles = ['car', 'truck', 'semi', 'motorcycle', None]

def traffic():
    while True:
        vehicle = choice(vehicles)
        if vehicle is None:
            return                          # 生成器函数终止不需要抛出StopIteration异常
        color = choice(colors)
        yield f'{color} {vehicle}'          # yield，声明为生成器函数，返回一个值，暂停执行，等待下次调用next()方法

count = 0
for count, vehicle in enumerate(traffic(), start=1):
    print(f"Wait for {vehicle}...")
print(f"Merged after {count} vehicles!")

- **生成器关闭：**
    - `close()`方法关闭生成器，在生成器暂停的`yield`语句处引发`GeneratorExit`，进而引发`StopIteration`结束循环。
    - 可以在生成器函数中使用`try/except`捕获`GeneratorExit`，并执行清理操作。
    - 关闭后的生成器不能再使用`next()`，否则抛出`StopIteration`。

In [ ]:
from random import choice
colors = ['red', 'green', 'blue', 'silver', 'white', 'black']
vehicles = ['car', 'truck', 'semi', 'motorcycle', None]

# 生成器函数
def traffic():
    while True:
        vehicle = choice(vehicles)
        color = choice(colors)
        try:
            yield f'{vehicle} {color}'
        except GeneratorExit:               # 捕获GeneratorExit异常
            print("No more vehicles.")
            raise                           # 必须重新引发！否则生成器不会真正关闭

def car_wash(traffic, limit):
    count = 0
    for vehicle in traffic:
        print(f"Washing {vehicle}.")
        count += 1
        if count >= limit:
            traffic.close()                 # 超过限制则关闭生成器

car_wash(traffic(), 3)

- **生成器抛出异常：**
    - `throw()`方法：可以在生成器空闲的`yield`语句处引发指定异常，用于让生成器进入某种特殊状态。
    - `throw()`功能上等同于`close()`抛`GeneratorExit`异常。
    - 如果`throw()`抛出的异常未被生成器处理，该异常会冒泡出来，**不会静默**。
    - 实际使用场景很少，通常有更简单的替代方案。

In [ ]:
from random import choice
colors = ['red', 'green', 'blue', 'silver', 'white', 'black']
vehicles = ['car', 'truck', 'semi', 'motorcycle', None]

# 生成器函数
def traffic():
    while True:
        vehicle = choice(vehicles)
        color = choice(colors)
        try:
            yield f'{vehicle} {color}'
        except ValueError:                              # 捕获ValueError异常, 跳过该车辆
            print(f"Skipping {color} {vehicle}...")
            continue
        except GeneratorExit:                           # 捕获GeneratorExit异常
            print("No more vehicles.")
            raise

def wash_vehicle(vehicle):
    if 'semi' in vehicle:
        raise ValueError("Cannot wash vehicle.")
    print(f"Washing {vehicle}...")

def car_wash(traffic, limit=5):
    count = 0
    for vehicle in traffic:
        try:
            wash_vehicle(vehicle)
        except Exception as e:
            traffic.throw(e)                            # 抛出异常, 继续生成下一个车辆
        else:
            count += 1

        if count >= limit:
            traffic.close()                             # 关闭生成器, 退出循环

car_wash(traffic(), 10)

- **委托子生成器：**
    - `yield from`：允许临时将控制权交给其他可迭代对象、生成器或协程，将它们的输出直接传递给调用者。

In [ ]:
from random import choice, randint
colors = ['red', 'green', 'blue', 'silver', 'white', 'black']
vehicles = ['car', 'truck', 'semi', None]

# 子生成器：生成自行车队
def biker_gang():
    for _ in range(randint(2, 5)):
        color = choice(colors)
        yield f'{color} 自行车'

# 主生成器：生成交通流
def traffic():
    while True:
        if randint(0, 10) == 10:
            yield from biker_gang()  # 将控制权交给子生成器，直到子生成器完成运行
            continue
        vehicle = choice(vehicles)
        color = choice(colors)
        yield f'{color} {vehicle}'

count = 0
for count, vehicle in enumerate(traffic(), 1):
    print(f'{count}: {vehicle}')
    if count == 20:
        break

### 2. 生成器表达式

- **生成器表达式**：生成器表达式是一个迭代器，将整个生成器逻辑封装到单个表达式，用**圆括号**包裹。
    - 生成器表达式是**惰性的**，可以用来处理大量数据而无须锁定程序。
    - 生成器表达式中，最左边`for`语句中的表达式会被立即求值，尽量在`for`中使用惰性可迭代对象(如`range()`、`product()`等)。
    - 多循环生成器表达式中，循环按**从外到内**的顺序列出，与嵌套`for`循环完全一致。
    - 生成器的条件过滤，简单`if`过滤放在被过滤的`for`之后，`if-else`必须用三元表达式，不能在生成器表达式中单独使用`if-else`。
    - 当嵌套生成器表达式变得过于复杂时，应直接使用普通生成器函数。可读性永远是第一位的。生成式超过2层嵌套 → 改用`for`循环或生成器函数。

In [ ]:
# 简单生成器表达式：本质上是循环语句的反转
plates = (f'ABC{i:03}' for i in range(100))
print(next(plates))

# 多循环生成器表达式：循环从外到内列出
from itertools import product
from string import ascii_uppercase as alphabet
plates = (f"{''.join(letters)}{i:03}" for letters in product(alphabet, repeat=3) for i in range(1000))
print(next(plates))

# 生成器的条件过滤：注意语句顺序，与正常嵌套顺序相同
plates = (f"{''.join(letters)}{i:03}" for letters in product(alphabet, repeat=3) if letters != ('G', 'O', 'V') for i in range(1000))
print(next(plates))

# if-else生成器表达式：必须用三元表达式模式，并且放在最前面
divis = (i if i % 3 == 0 else 'redacted' for i in range(1, 100))
print(next(divis))

# 嵌套生成器表达式：慎用，很难看懂逻辑
plates = (
    f"{letters}{i:03}"
    for letters in (
        ''.join(chars) for chars in product(alphabet, repeat=3)  # 嵌套生成器表达式
    )
    if letters != 'GOV'
    for i in range(1000)
)
print(next(plates))

### 3.推导式

- **推导式**：即时求值。
    - 列表推导式：`[表达式 for 变量 in 可迭代对象 if 条件]`。
    - 集合推导式：`{表达式 for 变量 in 可迭代对象 if 条件}`。自动去重。
    - 字典推导式：`{键: 值 for 变量 in 可迭代对象 if 条件}`。重复的键会被最后一个值覆盖。


### 4.简单协程

- **协程**：协程也是一种生成器，能按需使用数据，而不生产数据(不能使用next()方法)，且会耐心等待直至接收到数据。分为简单协程和原生协程(异步协程)。
    - 协程的调用：`协程对象 = 函数名(参数)`。
    - 协程的启动：`协程对象.send(None)`。
    - 协程的执行：`值 = 协程对象.send(参数)`。
    - 协程的关闭：`协程对象.close()`。
    - 注意：
        - 协程必须先启动使其运行到第一个`yield`语句，不预激则首次发送的值会丢失。
        - 生成器与协程的区别：`yield`语句出现的位置不同，协程中`yield`表达式被赋值给某个变量(接收数据)，而非单独产出数据。
        - **协程总是先`yield`返回值，再接收`send()`的新值。**

In [ ]:
from random import choice
colors = ['red', 'green', 'blue', 'silver', 'white', 'black']
vehicles = ['car', 'truck', 'semi', None]

# 协程：接收数据并处理
def color_counter(color):
    matches = 0
    while True:
        vehicle = yield matches     # 协程与生成器的区别：协程使用yield接受数据，然后分配给某个对象，同时返回一个值
        if color in vehicle:
            matches += 1

# 生成器：生成数据
def traffic():
    while True:
        vehicle = choice(vehicles)
        color = choice(colors)
        yield f'{color} {vehicle}'  # 生成器与协程的区别：生成器使用yield返回数据，可以调用next()函数获取下一个值

counter = color_counter('red')                          # 创建协程对象
counter.send(None)                                      # 启动协程(初次启动必须发送None)
for count, vehicle in enumerate(traffic(), start=1):
    if count < 100:
        matches = counter.send(vehicle)                 # 向协程发送数据并接收返回值
    else:
        counter.close()                                 # 关闭协程
        break
print(f'当前匹配数为{matches}')

In [ ]:
# 协程行为：协程在接收来自send()的新值之前总会产生一个值。赋值表达式的右侧先于左侧被计算。

# 执行顺序：
# 启动协程 → 协程前进到yield，产出ret初始值(None)
# send(0) → 0被接收赋值给recv → recv打印并存储到ret → 前进到yield → 产出ret(0) → send()返回0
# send(1) → 1被接收赋值给recv → recv打印并存储到ret → 前进到yield → 产出ret(1) → send()返回1
# 以此类推...

def coroutine():
    ret = None
    while True:
        print('...')                        # 2.初次启动输出提示    9.每次循环输出提示
        recv = yield ret                    # 3.初次启动返回ret值   6.接收数据保存到recv   10.返回ret值
        print(f'recv: {recv}')              # 7.输出接收的数据
        ret = recv                          # 8.将接收的数据赋值给ret

co = coroutine()
current = co.send(None)                     # 1.启动协程
print(f'Current(ret-S): {current}')         # 4.输出启动时返回的None

for i in range(5):
    current = co.send(i)                    # 5.发送数据给协程   11.接收返回值赋值给current
    print(f'Current(ret-{i}): {current}')   # 11.输出每次循环返回的current值
co.close()

### 5.本章小结

- **核心知识脉络树**

```text
生成器与推导式
│
├── 惰性求值
│   ├── 即时可迭代对象（eager）→ 大数据风险：MemoryError/崩溃
│   └── 惰性可迭代对象（lazy）→ 按需产出，处理大数据安全
│       └── 无限迭代器（itertools: count/cycle/repeat）→ ⚠️ 必须配break
│
├── 生成器函数
│   ├── yield关键字 → 暂停/恢复，返回生成器迭代器
│   ├── return → 自动引发StopIteration（Python 3.5+禁止手动raise）
│   ├── close() → 引发GeneratorExit → 可捕获做清理，必须重新引发
│   ├── throw(exc) → 在yield处抛异常（少用，通常有更简方案）
│   └── yield from → 委托给子生成器/可迭代对象，耗尽后控制权恢复
│
├── 生成器表达式
│   ├── 语法：(expr for x in iter if cond)
│   ├── 惰性，yield隐式，for从外到内
│   ├── ⚠️ 最左for即时求值
│   ├── ⚠️ if-else必须用三元表达式
│   ├── 嵌套生成器表达式（内层作为外层可迭代对象）
│   └── 危险：超过2层嵌套→难读/难调试→改用生成器函数
│
├── 推导式（即时求值）
│   ├── 列表推导式 [expr for x in iter if cond] → list
│   ├── 集合推导式 {expr for x in iter if cond} → set（自动去重）
│   └── 字典推导式 {k:v for x in iter if cond} → dict（键去重）
│
└── 简单协程
    ├── 本质：消费数据的特殊生成器（yield在赋值右侧接收值）
    ├── 启动：必须send(None)预激
    ├── send(value) → 发送值+返回yield的值
    ├── 返回值：vehicle = yield matches（同时接收+返回）
    └── 行为顺序：先yield返回值，再接收send()的新值
```

- **警告与提示表**

| 类型   | 内容                                                         |
| ------ | ------------------------------------------------------------ |
| ⚠️ 危险 | 无限迭代器必须配合`break`使用，否则程序卡死/崩溃             |
| ⚠️ 关键 | 生成器/生成器表达式**只能迭代一次**，再次迭代返回空          |
| ⚠️ 陷阱 | 生成器表达式**最左`for`是即时求值**的，其余惰性；`for x in [列表]`会立即求值 |
| ⚠️ 语法 | 生成器表达式中`if-else`**必须用三元表达式**`a if cond else b`，不能单独`if-else` |
| ⚠️ 必须 | 协程必须先`send(None)`预激，否则首次发送的值丢失             |
| ⚠️ 禁止 | Python 3.5+禁止在生成器函数中`raise StopIteration`，会引发`RuntimeError` |
| ⚠️ 必须 | 捕获`GeneratorExit`后**必须重新引发**，否则生成器不会真正关闭 |
| ⚠️ 性能 | 列表/集合/字典推导式是即时的，大数据集用生成器表达式避免内存爆炸 |
| ⚠️ 滥用 | 不能用推导式替代普通循环（结果隐式丢弃的模式是滥用的信号）   |
| 💡 技巧 | 简单逻辑用生成器表达式，复杂逻辑用生成器函数                 |
| 💡 技巧 | `yield from`简化子生成器委托，替代`for item in it: yield item` |
| 💡 技巧 | 推导式超过2层嵌套 → 改用`for`循环或生成器函数                |
| 💡 技巧 | 先用传统循环写第一版，确保正确后再压缩成推导式               |
| 💡 技巧 | 协程行为顺序：先`yield`返回值，再接收`send()`的新值          |